# 🧠 RetentionAI — End-to-End ML Pipeline

> **Hackathon Submission** | AI-powered Customer Churn Prediction & Retention Intelligence

---

### Pipeline Overview
| Step | Description |
|------|-------------|
| 1 | Synthetic data generation + EDA |
| 2 | Feature engineering |
| 3 | XGBoost training + hyperparameter tuning |
| 4 | SHAP explainability (global + per-customer) |
| 5 | AUC-ROC + Precision-Recall evaluation |
| 6 | BERT emotion detection demo |
| 7 | Digital twin counterfactual simulation |


In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

# Add project root to path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.style as style
import seaborn as sns

style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})

print('✅ Imports OK')

## 📂 Step 1 — Data Generation + EDA

In [ ]:
# Generate synthetic dataset
%run ../ml/generate_dataset.py

df = pd.read_csv(ROOT / 'data' / 'synthetic_customers.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# EDA — Churn distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Churn rate
churn_counts = df['churn'].value_counts()
axes[0].bar(['No Churn', 'Churn'], churn_counts.values,
            color=['#10b981', '#e11d48'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Churn Distribution')
axes[0].set_ylabel('Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

# Risk tier
tier_counts = df['risk_tier'].value_counts()
colors = {'high': '#e11d48', 'medium': '#d97706', 'low': '#10b981'}
axes[1].bar(tier_counts.index, tier_counts.values,
            color=[colors.get(t, '#6366f1') for t in tier_counts.index])
axes[1].set_title('Risk Tier Distribution')

# Churn prob distribution
axes[2].hist(df['churn_probability'], bins=30, color='#6366f1', alpha=0.7, edgecolor='white')
axes[2].axvline(0.38, color='#e11d48', linestyle='--', label='Decision threshold')
axes[2].set_title('Churn Probability Distribution')
axes[2].set_xlabel('Churn Probability')
axes[2].legend()

plt.suptitle('RetentionAI — EDA Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlations with churn
numeric_cols = ['age', 'tenure_months', 'product_count', 'avg_balance',
                'monthly_txn_count', 'complaints_open', 'nps_score',
                'competitor_inquiry', 'app_login_days', 'intl_transfer_dormancy']

corr = df[numeric_cols + ['churn']].corr()['churn'].drop('churn').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e11d48' if v > 0 else '#10b981' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Churn')
ax.set_title('Feature → Churn Correlation Analysis', fontsize=13)
plt.tight_layout()
plt.show()

print('Top positive correlators (churn drivers):')
print(corr[corr > 0].to_string())

## ⚙️ Step 2 — Feature Engineering

In [ ]:
FEATURES = [
    'age', 'tenure_months', 'product_count', 'avg_balance',
    'monthly_txn_count', 'clv_inr', 'complaints_open', 'nps_score',
    'competitor_inquiry', 'app_login_days', 'intl_transfer_dormancy',
    'emotion_label'
]

X = df[FEATURES].fillna(0)
y = df['churn']

print(f'Feature matrix shape: {X.shape}')
print(f'Class balance: {y.value_counts().to_dict()}')
X.describe().round(2)

## 🚀 Step 3 — XGBoost Training

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.08,
    subsample=0.85, colsample_bytree=0.85,
    scale_pos_weight=(y==0).sum()/(y==1).sum(),
    use_label_encoder=False, verbosity=0, random_state=42
)

# 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
print(f'5-Fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# Final training
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f'Test AUC: {auc:.4f}')
print(classification_report(y_test, model.predict(X_test), target_names=['No Churn', 'Churn']))

## 🔍 Step 4 — SHAP Explainability

In [ ]:
import shap

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Global SHAP — Beeswarm
plt.figure(figsize=(11, 7))
shap.summary_plot(shap_values, X_test, feature_names=FEATURES, show=False)
plt.title('Global Feature Impact (SHAP Beeswarm)', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# Per-customer waterfall — highest risk customer
high_risk_idx = y_proba.argmax()
print(f'Customer at index {high_risk_idx} | Churn Prob: {y_proba[high_risk_idx]:.2%}')

shap_exp = explainer(X_test)
shap.waterfall_plot(shap_exp[high_risk_idx], max_display=10, show=True)

## 📈 Step 5 — Model Evaluation Curves

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

avg_prec  = average_precision_score(y_test, y_proba)
fpr, tpr, _ = roc_curve(y_test, y_proba)
prec, rec, _ = precision_recall_curve(y_test, y_proba)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ROC
ax1.plot(fpr, tpr, '#6366f1', lw=2.5, label=f'AUC = {auc:.3f}')
ax1.plot([0,1],[0,1],'k--',lw=1, alpha=0.4)
ax1.fill_between(fpr, tpr, alpha=0.08, color='#6366f1')
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title('AUC-ROC Curve', fontsize=13); ax1.legend(fontsize=12)

# PR
ax2.plot(rec, prec, '#10b981', lw=2.5, label=f'AP = {avg_prec:.3f}')
ax2.fill_between(rec, prec, alpha=0.08, color='#10b981')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve', fontsize=13); ax2.legend(fontsize=12)

plt.suptitle('RetentionAI — Model Evaluation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Summary: AUC = {auc:.4f} | Average Precision = {avg_prec:.4f}')

## 🤖 Step 6 — BERT Emotion Detection Demo

In [ ]:
from backend.models.emotion_detector import EmotionDetector

detector = EmotionDetector()

test_texts = [
    'This app keeps crashing and my complaint has been unresolved for 2 weeks!',
    'I noticed HDFC is offering better rates. I am thinking of switching.',
    'I dont understand how to use the international transfer feature.',
    'The fees are too high. I want a discount or I will close my account.',
    'Really happy with the service! The RM was super helpful.',
]

results = [(text[:55]+'...', detector.classify(text)) for text in test_texts]
print(f'{'Text':<58} {'Emotion':<22} {'Confidence':>10}  Mode')
print('-' * 100)
for text, r in results:
    print(f'{text:<58} {r["emotion"]:<22} {r["confidence"]:>10.2%}  {r["mode"]}')

## 🧪 Step 7 — Digital Twin Counterfactual Simulation

In [ ]:
from backend.services.digital_twin import simulate_intervention

# Customer: Priya — 81% churn risk, Frustrated, CLV ₹4.2L
customer_ctx = {'emotion_name': 'Frustrated', 'clv_inr': 420000, 'tenure_months': 36}
base_score   = 0.81

print('\n🎯 Digital Twin Simulation — Priya Krishnamurthy')
print(f'   Base churn probability: {base_score:.0%}\n')

scenarios = ['none', 'offer', 'call', 'upgrade']
sim_results = {}

for action in scenarios:
    r = simulate_intervention(base_score, action, customer_ctx)
    sim_results[action] = r
    print(f'  [{action.upper():<9}] New score: {r["new_score"]:>5.1f}% | '
          f'Delta: {r["delta_pts"]:>8} | Revenue retained: {r["revenue_retained"]}')

In [ ]:
# Visualize simulation results
labels   = [s.upper() for s in scenarios]
scores   = [sim_results[s]['new_score'] for s in scenarios]
col_map  = {'NONE': '#94a3b8', 'OFFER': '#f59e0b', 'CALL': '#6366f1', 'UPGRADE': '#10b981'}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, scores,
              color=[col_map[l] for l in labels],
              width=0.5, edgecolor='white', linewidth=1.5)
ax.axhline(base_score*100, color='#e11d48', linestyle='--',
           label=f'Baseline: {base_score:.0%}', linewidth=1.5)

for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1.5,
            f'{score:.1f}%', ha='center', fontweight='bold', fontsize=12)

ax.set_ylabel('Churn Probability After Intervention (%)')
ax.set_title('🧪 Digital Twin — Intervention Scenarios', fontsize=13, fontweight='bold', pad=12)
ax.legend()
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

---
## ✅ Pipeline Complete

| Module | Status |
|--------|--------|
| Synthetic dataset (500 rows) | ✅ |
| XGBoost churn model | ✅ |
| SHAP explainability (global + per-customer) | ✅ |
| AUC-ROC + Precision-Recall evaluation | ✅ |
| BERT emotion detection | ✅ |
| Digital twin simulation | ✅ |

**Next:** Run `uvicorn backend.main:app --reload` and open the dashboard.
